In [ ]:
BAGIAN = "validasi"          # "validasi" sekarang; "uji" hanya sekali, dengan BUKA_UJI=1 (lihat sel pengaman)

# Pemodelan 03 — Evaluasi Akhir

Notebook ini menghitung metrik, tabel, dan gambar akhir untuk satu bagian data, dipilih dengan
`BAGIAN` di sel pertama. Sekarang `BAGIAN = "validasi"`: semua angka harus sama dengan tahap D dan
`uji_tau.csv` di notebook 02 (assert). Untuk bagian uji cukup `BAGIAN = "uji"`, dijalankan **sekali**
dengan `BUKA_UJI=1`; tidak ada model yang dilatih ulang dan tidak ada pilihan yang diubah.

| | |
|---|---|
| Data | Tensor v2 R3_titik: V17, jendela 8 hari, 11 kanal per pos, log1p |
| Model | Persistensi; ARIMA(2, 0, 3) dengan orde tetap dari AIC (notebook 01); CNN tanh-Glorot; LSTM; LSTM-MC Dropout; CNN-LSTM T21; CNN-LSTM-MC Dropout |
| Bobot | Bobot tersimpan di `cache/bobot/` (tidak masuk git); CNN tanh-Glorot dipulihkan dengan latih ulang yang harus identik bit dengan run tercatat |
| MC Dropout | Bentuk B (tahap 21–22): 1.000 lintasan; τ⁻¹ dari residu bagian latih yang sudah tercatat; ±1σ = rata-rata ± √(sd² + τ⁻¹); 95% = persentil 2,5–97,5% lintasan + derau N(0, τ⁻¹); prediksi titik = rata-rata lintasan |
| Seed | 0–7 untuk setiap model jaringan; metrik dilaporkan sebagai rata-rata ± sb antar-seed |
| A:e | Sampel uji yang targetnya berkode A:e (kode bagian 3) dikeluarkan; di validasi hanya dilaporkan jumlahnya |
| Kondisi aliran | < Q1, Q1–Q3, > Q3; Q1 dan Q3 = kuartil debit pos target bagian latih |
| Perangkat | Harus sama dengan perangkat run tercatat (NVIDIA GeForce RTX 4060) |
| Keluaran | `modeling/hasil/evaluasi_<bagian>_*.csv`, `figur/eval_<bagian>_*.png` (300 dpi, label bahasa Inggris); lintasan MC di `cache/evaluasi_<bagian>/` |

Dasar: [Gal2016] untuk MC Dropout; bentuk B dan aturan pemilihannya: notebook 02 tahap 21–22;
[Box2015] dan [Akaike1974] untuk ARIMA; [Sun2026] (Fig. 5 dan 8) dan [Le2025] (Fig. 7) untuk bentuk gambar.

Menjalankan (Ubuntu, RTX 4060; lihat `modeling/README.md`):

```bash
.venv/bin/python -m jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=-1 modeling/03_evaluasi.ipynb
```

## Persiapan

In [ ]:
import hashlib, json, os, subprocess, sys, time, warnings
from datetime import datetime
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

warnings.filterwarnings("ignore")
MULAI_JALAN = time.time()

# --- lokasi folder: notebook bisa dijalankan dari root repo maupun dari folder modeling/
ROOT = Path.cwd() if (Path.cwd() / "processed").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "modeling"))
import pemodelan as pm
from pemodelan import ke_m3s

assert BAGIAN in ("validasi", "uji"), f"BAGIAN harus 'validasi' atau 'uji', bukan {BAGIAN!r}"
KELUARAN, FIGUR = pm.HASIL, pm.FIGUR                 # tabel CSV dan gambar
PERIODE, HORIZON, TARGET = pm.PERIODE, pm.HORIZON, pm.TARGET
I_LATIH, I_VAL = pm.I_LATIH, pm.I_VAL
SEED = list(range(8))
N_LINTASAN = pm.N_LINTASAN
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print(f"BAGIAN = {BAGIAN!r}; batas latih/validasi {PERIODE[I_LATIH].date()}, validasi/uji {PERIODE[I_VAL].date()}")

## 0. Pengaman bagian uji

Bila `BAGIAN = "uji"`, notebook berhenti di sini kecuali variabel lingkungan `BUKA_UJI=1` diset
**dan** `modeling/hasil/uji_dibuka.txt` belum ada. Berkas itu ditulis tepat sebelum data uji dimuat
(sel 2), berisi waktu, hash commit, dan hash konfigurasi evaluasi, sehingga bagian uji hanya bisa
dibuka sekali. Pada `BAGIAN = "validasi"`, kedua fungsi pemuat bagian uji diganti fungsi yang selalu
gagal, dan setiap indeks yang dimuat diperiksa < awal bagian uji.

In [ ]:
UJI_DIBUKA = pm.HASIL / "uji_dibuka.txt"

def pengaman_uji(bagian, berkas, env):
    """Berhenti bila bagian uji diminta tanpa BUKA_UJI=1, atau bila uji sudah pernah dibuka."""
    if bagian != "uji":
        return
    if env.get("BUKA_UJI") != "1":
        raise RuntimeError("BAGIAN = 'uji' tetapi BUKA_UJI=1 tidak diset: notebook berhenti")
    if berkas.exists():
        raise RuntimeError(f"bagian uji sudah pernah dibuka, uji hanya sekali:\n{berkas.read_text()}")

pengaman_uji(BAGIAN, UJI_DIBUKA, os.environ)

if BAGIAN == "validasi":
    def _uji_ditutup(*a, **k):
        raise AssertionError("PAGAR: pemuat bagian uji dipanggil pada BAGIAN = 'validasi'")
    pm.muat_uji = pm.muat_uji_v2 = pm.muat_deret_target_semua = _uji_ditutup
print(f"pengaman: BAGIAN = {BAGIAN!r}; uji_dibuka.txt {'ada' if UJI_DIBUKA.exists() else 'belum ada'}")

## 1. Konfigurasi evaluasi

Semua yang menentukan angka dikumpulkan di satu kamus `KONFIG_EVAL`: data, hiperparameter T21,
`id_run` setiap model × horizon × seed, τ⁻¹ tercatat, benih, orde ARIMA, Q1 dan Q3. Hash SHA-256-nya
disimpan di `evaluasi_<bagian>_konfigurasi.json`; pada bagian uji hash ini harus sama dengan hash
validasi, sehingga konfigurasi tidak bisa berubah di antara keduanya.

- CNN-LSTM: run `B_final_R3` (tahap 17); τ⁻¹ dan benih derau dari `uji_tau.csv` (tahap 21).
- LSTM: run `D_banding` (tahap 23); τ⁻¹ dan benih derau dari `tahap_d_per_run.csv`.
- CNN tanh-Glorot: run `D_cnn_glorot` (tahap 24), tanpa MC.
- Benih MC = seed run; `keras.utils.set_random_seed(seed)` juga dipanggil sebelum model dibangun,
  sehingga lintasan tidak bergantung pada urutan run.

In [ ]:
KONF_DATA = dict(varian="V17", jendela=8, hujan="v2_R3_titik", transformasi="log1p")
CFG_T21 = json.loads((pm.HASIL / "pilihan_tahap_b.json").read_text())["model"]["cnn_lstm"]["cfg"]
assert json.loads((pm.HASIL / "pilihan_tahap_c.json").read_text())["terpilih"] == "B"
UJI_TAU = pd.read_csv(pm.HASIL / "uji_tau.csv")
TD_REF = pd.read_csv(pm.HASIL / "tahap_d_per_run.csv")

def daftar_run(t):
    """{(horizon, seed): id_run} untuk 3 horizon x 8 seed."""
    d = {(int(r.horizon), int(r.seed)): r.id_run for r in t.itertuples()}
    assert sorted(d) == [(h, s) for h in HORIZON for s in SEED], "run tidak lengkap"
    return d

RUN = {"cnn_lstm": daftar_run(UJI_TAU),
       "lstm": daftar_run(TD_REF[TD_REF["model"] == "LSTM"]),
       "cnn_tanh_glorot": daftar_run(TD_REF[TD_REF["model"] == "CNN"])}
assert daftar_run(TD_REF[TD_REF["model"] == "CNN-LSTM tanpa MC"]) == RUN["cnn_lstm"]
lstm_mc = TD_REF[TD_REF["model"] == "LSTM-MC Dropout"]
assert daftar_run(lstm_mc) == RUN["lstm"]
TAU = {"cnn_lstm": {(int(r.horizon), int(r.seed)): float(r.tau_inv_m3s2) for r in UJI_TAU.itertuples()},
       "lstm": {(int(r.horizon), int(r.seed)): float(r.tau_inv_m3s2) for r in lstm_mc.itertuples()}}
BENIH_DERAU = {"cnn_lstm": lambda h, s: 2016_000 + 100 * h + s,      # tahap 21
               "lstm": lambda h, s: 2018_000 + 100 * h + s}          # tahap 23
for kode_, t_ in [("cnn_lstm", UJI_TAU), ("lstm", lstm_mc)]:
    assert all(int(r.benih_derau) == BENIH_DERAU[kode_](int(r.horizon), int(r.seed)) for r in t_.itertuples())

log_ = pm.baca_log().set_index("id_run")
semua_id = [i for d in RUN.values() for i in d.values()]
MESIN_RUN = log_.loc[semua_id, "mesin"].unique()
assert len(MESIN_RUN) == 1, f"run berasal dari lebih dari satu perangkat: {MESIN_RUN}"
MESIN_RUN = str(MESIN_RUN[0])
for kode_, d in RUN.items():
    assert (log_.loc[list(d.values()), "model"] == kode_).all()
    assert (log_.loc[list(d.values()), "konfigurasi"] == pm.konfig_json(kode_, CFG_T21)).all()
assert pm.mesin() == MESIN_RUN, f"perangkat {pm.mesin()} berbeda dengan perangkat run tercatat {MESIN_RUN}"

q_latih = pm.muat_deret_target().iloc[:I_LATIH]                  # kuartil dari bagian latih saja
Q1, Q3 = (float(v) for v in q_latih.quantile([0.25, 0.75]))
assert (round(Q1, 1), round(Q3, 1)) == (57.8, 220.9)
ORDE_ARIMA, TREN_ARIMA = (2, 0, 3), "c"                           # AIC di notebook 01, dipakai ulang di tahap 23

KONFIG_EVAL = {"data": KONF_DATA, "cfg_t21": CFG_T21, "tetap": pm.TETAP, "seed": SEED, "n_lintasan": N_LINTASAN,
               "run": {k: {f"h{h}_s{s}": i for (h, s), i in d.items()} for k, d in RUN.items()},
               "tau_inv_m3s2": {k: {f"h{h}_s{s}": v for (h, s), v in d.items()} for k, d in TAU.items()},
               "benih_mc": "seed run", "benih_derau": {"cnn_lstm": "2016000 + 100*h + seed",
                                                       "lstm": "2018000 + 100*h + seed"},
               "arima": {"orde": list(ORDE_ARIMA), "tren": TREN_ARIMA, "fit": "bagian latih, log-debit"},
               "Q1_latih_m3s": Q1, "Q3_latih_m3s": Q3, "mesin": MESIN_RUN}
HASH_KONFIG = hashlib.sha256(json.dumps(KONFIG_EVAL, sort_keys=True).encode()).hexdigest()
(KELUARAN / f"evaluasi_{BAGIAN}_konfigurasi.json").write_text(
    json.dumps({"hash_konfigurasi": HASH_KONFIG, **KONFIG_EVAL}, indent=1, ensure_ascii=False))
print(f"perangkat: {MESIN_RUN}; Q1 {Q1:.1f} m3/s, Q3 {Q3:.1f} m3/s; ARIMA{ORDE_ARIMA}")
print(f"hash konfigurasi: {HASH_KONFIG}")

## 2. Data bagian yang dievaluasi

Validasi dimuat dengan `muat_latih_val` (pagar data uji di dalamnya). Bagian uji dimuat dengan
`muat_uji_v2`, yang juga menolak berjalan sebelum `konfigurasi_final.json` dikunci. Sebelum data uji
dimuat, semua prasyarat diperiksa dulu (konfigurasi final, hash konfigurasi sama dengan validasi,
bobot lengkap, `pemodelan.py` tanpa perubahan yang belum di-commit), lalu `uji_dibuka.txt` ditulis.

In [ ]:
def git(*a):
    return subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True, check=True).stdout.strip()

if BAGIAN == "uji":
    pengaman_uji(BAGIAN, UJI_DIBUKA, os.environ)                      # diperiksa ulang tepat sebelum dibuka
    pm._cek_konfig_final()
    acuan = json.loads((KELUARAN / "evaluasi_validasi_konfigurasi.json").read_text())
    assert acuan["hash_konfigurasi"] == HASH_KONFIG, "konfigurasi evaluasi berbeda dengan versi validasi"
    hilang = [i for k, d in RUN.items() for i in d.values() if not (pm.BOBOT / f"{i}.weights.h5").exists()]
    assert not hilang, f"bobot belum lengkap ({len(hilang)}): jalankan dulu dengan BAGIAN = 'validasi'"
    assert not git("status", "--porcelain", "--", "modeling/pemodelan.py"), "pemodelan.py punya perubahan belum di-commit"
    commit, kotor = git("rev-parse", "HEAD"), git("status", "--porcelain")
    with open(UJI_DIBUKA, "x") as f:                                  # mode "x": gagal bila berkas sudah ada
        f.write(f"waktu: {datetime.now().isoformat(timespec='seconds')}\ncommit: {commit}\n"
                f"hash_konfigurasi: {HASH_KONFIG}\nmesin: {MESIN_RUN}\n"
                f"perubahan_belum_commit: {kotor.replace(chr(10), '; ') or 'tidak ada'}\n")
    print(UJI_DIBUKA.read_text())

DATA, DATA_LV = {}, {}
baris_n = []
for h in HORIZON:
    if BAGIAN == "validasi":
        D = pm.muat_latih_val(h, **KONF_DATA)                          # pagar data uji di dalam fungsi ini
        DATA_LV[h] = D
        X, y, idx, murni = D["X_val"], D["y_val"], D["idx_val"], D["murni_val"]
        assert (D["idx_latih"] < I_LATIH).all() and (idx >= I_LATIH).all() and (idx < I_VAL).all()
        pakai = np.ones(len(y), bool)                                  # validasi: A:e hanya dihitung
    else:
        D = pm.muat_uji_v2(h, "R3_titik")
        X, y, idx, murni = D["X_uji"], D["y_uji"], D["idx_uji"], D["murni_uji"]
        assert (idx >= I_VAL).all()
        pakai = murni                                                  # uji: A:e dikeluarkan
    DATA[h] = dict(h=h, X=X[pakai], y=y[pakai], idx=idx[pakai], rata=D["rata"], std=D["std"], i_target=D["i_target"])
    DATA[h]["obs"] = ke_m3s(DATA[h]["y"], DATA[h])
    DATA[h]["tanggal"] = PERIODE[DATA[h]["idx"]]
    baris_n.append({"horizon": h, "sampel": len(y), "target A:e": int((~murni).sum()),
                    "dikeluarkan": int((~pakai).sum()), "dievaluasi": int(pakai.sum()),
                    "< Q1": int((DATA[h]["obs"] < Q1).sum()), "Q1–Q3": int(((DATA[h]["obs"] >= Q1) & (DATA[h]["obs"] <= Q3)).sum()),
                    "> Q3": int((DATA[h]["obs"] > Q3).sum()),
                    "tanggal target": f"{DATA[h]['tanggal'][0].date()} – {DATA[h]['tanggal'][-1].date()}"})
if BAGIAN == "validasi":
    dimuat = [d["idx"] for d in DATA.values()] + [d["idx_latih"] for d in DATA_LV.values()] + [d["idx_val"] for d in DATA_LV.values()]
    assert max(int(i.max()) for i in dimuat) < I_VAL and q_latih.index[-1] < PERIODE[I_VAL], \
        "PAGAR: ada indeks bagian uji yang dimuat pada BAGIAN = 'validasi'"
    print("assert: tidak ada indeks uji yang dimuat (semua indeks < awal bagian uji)")
N_TABEL = pd.DataFrame(baris_n).set_index("horizon")
display(N_TABEL)

## 3. Bobot CNN tanh-Glorot

CNN tanh-Glorot (tahap 24) dilatih tanpa penyimpanan bobot. Pada `BAGIAN = "validasi"`, bobotnya
dipulihkan dengan `pm.pulihkan_bobot`: latih ulang dengan seed yang sama di perangkat yang sama, lalu
assert val_loss setiap epoch dan prediksi validasi identik bit dengan run tercatat sebelum bobot
disimpan. Run yang bobotnya sudah ada dilewati. Pada bagian uji tidak ada pelatihan sama sekali.

In [ ]:
t0, n_pulih = time.perf_counter(), 0
if BAGIAN == "validasi":
    for (h, s), id_ in sorted(RUN["cnn_tanh_glorot"].items()):
        if not (pm.BOBOT / f"{id_}.weights.h5").exists():
            pm.pulihkan_bobot("cnn_tanh_glorot", CFG_T21, DATA_LV[h], s, id_)
            n_pulih += 1
hilang = [i for d in RUN.values() for i in d.values() if not (pm.BOBOT / f"{i}.weights.h5").exists()]
assert not hilang, f"bobot tidak ada: {hilang[:3]} ... ({len(hilang)})"
print(f"bobot CNN dipulihkan: {n_pulih} run ({(time.perf_counter() - t0) / 60:.1f} menit); "
      f"bobot lengkap untuk {sum(len(d) for d in RUN.values())} run")

## 4. Model jaringan: deterministik dan MC Dropout bentuk B

Setiap run: bangun model, muat bobot, prediksi dengan dropout nonaktif. Pada validasi, prediksi ini
harus identik bit dengan prediksi tersimpan (`prediksi_val/`), sebagai bukti bobotnya benar. Untuk LSTM
dan CNN-LSTM, 1.000 lintasan MC Dropout dihitung dengan benih = seed run dan disimpan (float32) di
`cache/evaluasi_<bagian>/` bersama benih MC dan benih derau; bila berkasnya sudah ada, lintasan dibaca
ulang. Bentuk B dihitung seperti tahap 23.

In [ ]:
import keras, tensorflow as tf
tf.get_logger().setLevel("ERROR")
MC_CACHE = ROOT / "cache" / f"evaluasi_{BAGIAN}"
MC_CACHE.mkdir(parents=True, exist_ok=True)
LABEL_DET = {"cnn_tanh_glorot": "CNN", "lstm": "LSTM", "cnn_lstm": "CNN-LSTM"}
LABEL_MC = {"lstm": "LSTM-MC", "cnn_lstm": "CNN-LSTM-MC"}
URUTAN = ["Persistence", "ARIMA", "CNN", "LSTM", "LSTM-MC", "CNN-LSTM", "CNN-LSTM-MC"]
MODEL_MC = ["LSTM-MC", "CNN-LSTM-MC"]

def prediksi_jaringan(kode, h, s, id_run):
    """Prediksi deterministik (m3/s) dan, untuk model ber-MC, matriks lintasan (lintasan x sampel, m3/s, float32)."""
    E = DATA[h]
    X = pm.bentuk_masukan(E["X"], kode, E["i_target"])
    keras.backend.clear_session()
    keras.utils.set_random_seed(s)
    m = pm.bangun_model(kode, CFG_T21, X.shape[1:])
    m.load_weights(pm.BOBOT / f"{id_run}.weights.h5")
    pred = ke_m3s(m.predict(X, batch_size=2048, verbose=0).ravel(), E)
    if BAGIAN == "validasi":
        with np.load(pm.PREDIKSI / f"{id_run}.npz") as f:
            assert np.array_equal(f["idx_val"], E["idx"]) and np.array_equal(f["prediksi"], pred), \
                f"bobot {id_run} tidak menghasilkan prediksi validasi tersimpan"
    if kode not in LABEL_MC:
        return pred, None
    f_mc = MC_CACHE / f"{id_run}.npz"
    if f_mc.exists():
        with np.load(f_mc) as f:
            assert np.array_equal(f["idx"], E["idx"]) and int(f["benih_mc"]) == s and f["hash_konfigurasi"] == HASH_KONFIG
            S = f["lintasan"]
    else:
        S = ke_m3s(pm.mc_dropout(m, X, N_LINTASAN, s), E).astype(np.float32)
        np.savez_compressed(f_mc, lintasan=S, idx=E["idx"], benih_mc=s, benih_derau=BENIH_DERAU[kode](h, s),
                            tau_inv_m3s2=TAU[kode][(h, s)], bagian=BAGIAN, hash_konfigurasi=HASH_KONFIG)
    return pred, S

def bentuk_b(S, tau_inv, benih):
    """Bentuk B dari lintasan S (lintasan x sampel, m3/s), sama dengan metrik_b tahap 23."""
    S = S.astype(np.float64)
    mu, sd = S.mean(0), S.std(0)
    SB = S + np.random.default_rng(benih).normal(0.0, np.sqrt(tau_inv), size=S.shape)
    lo, hi = np.quantile(SB, [0.025, 0.975], axis=0)
    return dict(mu=mu, sdB=np.sqrt(sd ** 2 + tau_inv), lo95=lo, hi95=hi, persen_negatif=float((SB < 0).mean() * 100))

In [ ]:
HASIL_RUN = {}            # (model, horizon, seed) -> dict(pred, dan untuk model ber-MC: mu, sdB, lo95, hi95)
INFO_RUN = {}             # (model, horizon, seed) -> id_run, tau, benih
t0 = time.perf_counter()
for kode, d in RUN.items():
    for (h, s), id_ in sorted(d.items()):
        pred, S = prediksi_jaringan(kode, h, s, id_)
        HASIL_RUN[(LABEL_DET[kode], h, s)] = dict(pred=pred)
        INFO_RUN[(LABEL_DET[kode], h, s)] = dict(id_run=id_)
        if S is not None:
            b = bentuk_b(S, TAU[kode][(h, s)], BENIH_DERAU[kode](h, s))
            HASIL_RUN[(LABEL_MC[kode], h, s)] = dict(pred=b["mu"], **b)
            INFO_RUN[(LABEL_MC[kode], h, s)] = dict(id_run=id_, tau_inv_m3s2=TAU[kode][(h, s)], benih_mc=s,
                                                     benih_derau=BENIH_DERAU[kode](h, s))
            del S
    print(f"{LABEL_DET[kode]:9}: {len(d)} run selesai ({(time.perf_counter() - t0) / 60:.1f} menit sejak awal)")
if BAGIAN == "validasi":
    print("assert: prediksi deterministik dari bobot tersimpan identik dengan prediksi validasi tercatat (72 run)")
# informasi: lintasan baru dibanding lintasan tahap 21 / 23 (bila cache lama ada di mesin ini)
for kode, lama in [("cnn_lstm", "mc_val_tau"), ("lstm", "mc_val_lstm")]:
    beda = []
    for (h, s), id_ in RUN[kode].items():
        f_lama = ROOT / "cache" / lama / f"{id_}.npz"
        if BAGIAN == "validasi" and f_lama.exists():
            with np.load(f_lama) as a, np.load(MC_CACHE / f"{id_}.npz") as b_:
                beda.append(float(np.abs(a["lintasan"].astype(np.float64) - b_["lintasan"]).max()))
    if beda:
        print(f"{LABEL_MC[kode]}: selisih mutlak maks lintasan baru vs cache/{lama}: {max(beda):.3g} m3/s ({len(beda)} run)")

## 5. ARIMA dan persistensi

ARIMA memakai log-debit pos target saja, dengan orde tetap ARIMA(2, 0, 3) dan tren konstan dari
pemilihan AIC notebook 01. Parameter di-fit pada bagian latih, lalu dikunci; filter Kalman dijalankan
pada deret sampai akhir bagian yang dievaluasi (validasi: hanya sampai hari terakhir validasi).
Prakiraan h hari dari hari t memakai keadaan terprediksi sesudah hari t, sama dengan tahap 23.
Rentang 95% ARIMA (hanya untuk Gambar c, h1) dari ragam prakiraan satu langkah model ruang keadaan,
dihitung di skala log lalu dikembalikan ke m³/s. Persistensi = debit pos target hari t.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
t0 = time.perf_counter()
q_arima = pm.muat_deret_target() if BAGIAN == "validasi" else pm.muat_deret_target_semua()
lq = np.log(q_arima.to_numpy())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res_latih = SARIMAX(lq[:I_LATIH], order=ORDE_ARIMA, trend=TREN_ARIMA).fit(disp=False, maxiter=1000)
    res_semua = SARIMAX(lq, order=ORDE_ARIMA, trend=TREN_ARIMA).filter(res_latih.params)   # parameter dikunci
ssm = res_semua.model.ssm
def matriks(nama, dim):
    M = np.asarray(ssm[nama])
    return M[..., 0] if M.ndim > dim else M
T_, c_, Z_, d_, H_ = (matriks("transition", 2), matriks("state_intercept", 1), matriks("design", 2),
                      matriks("obs_intercept", 1), matriks("obs_cov", 2))
a_ramal, P_ramal = res_semua.predicted_state, res_semua.predicted_state_cov

def prakiraan_arima(asal, h):
    a = a_ramal[:, asal + 1]
    for _ in range(h - 1):
        a = c_[:, None] + T_ @ a
    return (d_[:, None] + Z_ @ a).ravel()

for h, E in DATA.items():
    asal = E["idx"] - h
    assert (asal + 1 < len(lq)).all()
    p_log = prakiraan_arima(asal, h)
    HASIL_RUN[("ARIMA", h, 0)] = dict(pred=np.exp(p_log))
    if h == 1:                                       # rentang 95% satu langkah, skala log
        sd_log = np.sqrt(np.einsum("ik,kln,jl->n", Z_, P_ramal[:, :, asal + 1], Z_) + H_[0, 0])
        HASIL_RUN[("ARIMA", h, 0)] |= dict(lo95=np.exp(p_log - 1.959964 * sd_log), hi95=np.exp(p_log + 1.959964 * sd_log))
    HASIL_RUN[("Persistence", h, 0)] = dict(pred=ke_m3s(E["X"][:, -1, E["i_target"], 0], E))
print(f"ARIMA{ORDE_ARIMA}, tren '{TREN_ARIMA}', {len(res_latih.params)} parameter dari bagian latih "
      f"({len(lq[:I_LATIH])} hari); filter sampai {q_arima.index[-1].date()}; {time.perf_counter() - t0:.0f} detik")

## 6. Metrik

Untuk setiap run: NSE, RMSE, dan MAE untuk seluruh hari dan untuk tiga kondisi aliran menurut debit
terukur (< Q1, Q1–Q3, > Q3; di subkelompok, NSE terhadap rata-rata subkelompok itu). Untuk model
ber-MC juga PICP dan MPIW ±1σ dan 95%, total dan per kondisi aliran. Model jaringan: rata-rata ± sb
antar 8 seed; ARIMA dan persistensi deterministik (satu nilai).

In [ ]:
KONDISI = {"all": lambda o: np.ones(len(o), bool), "<Q1": lambda o: o < Q1,
           "Q1-Q3": lambda o: (o >= Q1) & (o <= Q3), ">Q3": lambda o: o > Q3}

def metrik_run(o, r):
    out = {}
    for k, f in KONDISI.items():
        m_, sfx = f(o), "" if k == "all" else f"_{k}"
        p = r["pred"][m_]
        out |= {f"n{sfx}": int(m_.sum()), f"NSE{sfx}": float(pm.nse(o[m_], p)), f"RMSE{sfx}": pm.rmse(o[m_], p),
                f"MAE{sfx}": pm.mae(o[m_], p)}
        if "sdB" in r:
            mu, sdB, lo, hi = r["mu"][m_], r["sdB"][m_], r["lo95"][m_], r["hi95"][m_]
            om = o[m_]
            out |= {f"PICP68{sfx}": float(np.mean((om >= mu - sdB) & (om <= mu + sdB))), f"MPIW68{sfx}": float(np.mean(2 * sdB)),
                    f"PICP95{sfx}": float(np.mean((om >= lo) & (om <= hi))), f"MPIW95{sfx}": float(np.mean(hi - lo))}
    if "persen_negatif" in r:
        out["persen_lintasan_negatif"] = r["persen_negatif"]
    return out

baris = []
for (mod, h, s), r in HASIL_RUN.items():
    baris.append({"bagian": BAGIAN, "model": mod, "horizon": h, "seed": s if mod not in ("ARIMA", "Persistence") else np.nan,
                  **INFO_RUN.get((mod, h, s), {}), **metrik_run(DATA[h]["obs"], r)})
PER_RUN = pd.DataFrame(baris)
PER_RUN["model"] = pd.Categorical(PER_RUN["model"], URUTAN, ordered=True)
PER_RUN = PER_RUN.sort_values(["model", "horizon", "seed"]).reset_index(drop=True)
assert PER_RUN.groupby("model", observed=True).size().to_dict() == {m: (3 if m in ("ARIMA", "Persistence") else 24) for m in URUTAN}
PER_RUN.round(6).to_csv(KELUARAN / f"evaluasi_{BAGIAN}_per_run.csv", index=False)
RATA = PER_RUN.drop(columns=[c for c in ["bagian", "id_run", "seed", "benih_mc", "benih_derau", "tau_inv_m3s2"] if c in PER_RUN]) \
    .groupby(["model", "horizon"], observed=True).agg(["mean", "std"])
print(f"{len(PER_RUN)} baris per run -> evaluasi_{BAGIAN}_per_run.csv")

## 7. Kesesuaian dengan tahap D dan `uji_tau.csv` (validasi saja)

Pada validasi, angka notebook ini harus sama dengan notebook 02, dengan toleransi 0,002 untuk NSE dan
PICP:

- model deterministik (persistensi, ARIMA, CNN, LSTM, CNN-LSTM): NSE **per run** terhadap
  `tahap_d_per_run.csv`, dan NSE CNN-LSTM per run terhadap `uji_tau.csv`;
- model ber-MC: rata-rata 8 seed per horizon untuk NSE, PICP ±1σ, dan PICP 95% terhadap
  `tahap_d_per_run.csv`, dan untuk CNN-LSTM-MC juga PICP bentuk B terhadap `uji_tau.csv`. Lintasan
  dibangkitkan ulang dari bobot, sehingga per run boleh sedikit berbeda (aturan tahap 21); selisih per
  run terbesar ikut dilaporkan.

Bila ada selisih di atas toleransi, notebook berhenti dan menampilkan tabel selisihnya.

In [ ]:
TOL = 0.002
NAMA_TD = {"Persistence": "Persistensi", "ARIMA": "ARIMA", "CNN": "CNN", "LSTM": "LSTM", "LSTM-MC": "LSTM-MC Dropout",
           "CNN-LSTM": "CNN-LSTM tanpa MC", "CNN-LSTM-MC": "CNN-LSTM MC Dropout"}
if BAGIAN == "validasi":
    cek = []
    ref = TD_REF.assign(seed=TD_REF["seed"].fillna(0).astype(int))
    kini = PER_RUN.assign(seed=PER_RUN["seed"].fillna(0).astype(int), model=PER_RUN["model"].astype(str))
    for mod in URUTAN:
        a = kini[kini["model"] == mod].set_index(["horizon", "seed"])
        b = ref[ref["model"] == NAMA_TD[mod]].set_index(["horizon", "seed"])
        assert a.index.sort_values().equals(b.index.sort_values()), f"run {mod} tidak sama dengan tahap D"
        kolom = ["NSE"] + (["PICP68", "PICP95"] if mod in MODEL_MC else [])
        for k in kolom:
            d_run = (a[k] - b.loc[a.index, k]).abs()
            if mod in MODEL_MC:
                d_rata = (a[k].groupby("horizon").mean() - b[k].groupby("horizon").mean()).abs()
                cek.append({"acuan": "tahap_d_per_run.csv", "model": mod, "metrik": k, "uji": "rata-rata 8 seed per horizon",
                            "selisih_maks": float(d_rata.max()), "selisih_per_run_maks": float(d_run.max())})
            else:
                cek.append({"acuan": "tahap_d_per_run.csv", "model": mod, "metrik": k, "uji": "per run",
                            "selisih_maks": float(d_run.max()), "selisih_per_run_maks": float(d_run.max())})
    ut = UJI_TAU.set_index(["horizon", "seed"])
    a = kini[kini["model"] == "CNN-LSTM"].set_index(["horizon", "seed"])
    d_ = (a["NSE"] - ut.loc[a.index, "NSE"]).abs()
    cek.append({"acuan": "uji_tau.csv", "model": "CNN-LSTM", "metrik": "NSE", "uji": "per run",
                "selisih_maks": float(d_.max()), "selisih_per_run_maks": float(d_.max())})
    a = kini[kini["model"] == "CNN-LSTM-MC"].set_index(["horizon", "seed"])
    for k in ["PICP68", "PICP95"]:
        d_run = (a[k] - ut.loc[a.index, f"{k}_B"]).abs()
        d_rata = (a[k].groupby("horizon").mean() - ut[f"{k}_B"].groupby("horizon").mean()).abs()
        cek.append({"acuan": "uji_tau.csv", "model": "CNN-LSTM-MC", "metrik": f"{k} (bentuk B)", "uji": "rata-rata 8 seed per horizon",
                    "selisih_maks": float(d_rata.max()), "selisih_per_run_maks": float(d_run.max())})
    CEK = pd.DataFrame(cek)
    CEK["lolos"] = CEK["selisih_maks"] <= TOL
    display(CEK.style.format({"selisih_maks": "{:.2e}", "selisih_per_run_maks": "{:.2e}"}))
    CEK.to_csv(KELUARAN / "evaluasi_validasi_cek_tahap_d.csv", index=False)
    assert CEK["lolos"].all(), f"angka validasi berbeda dengan tahap D / uji_tau.csv (> {TOL}): berhenti\n" \
        + CEK[~CEK["lolos"]].to_string()
    print(f"assert: semua {len(CEK)} pemeriksaan selisih <= {TOL}")
else:
    print("bagian uji: tidak ada acuan; pemeriksaan kesesuaian hanya pada validasi")

## 8. Tabel

Rata-rata ± sb antar 8 seed (ARIMA dan persistensi: satu nilai). CSV memuat rata-rata dan sb sebagai
kolom angka terpisah; tampilan di bawah memakai label bahasa Inggris.

- **Tabel A** (tabel utama, bentuk Tabel 2 [Sun2026]): baris Persistence (reference), ARIMA, CNN, LSTM,
  LSTM-MC Dropout, CNN-LSTM-MC Dropout; kolom NSE, RMSE, MAE, PICP95, MPIW95 per horizon. PICP95 dan
  MPIW95 diisi "—" untuk model tanpa MC. Prediksi titik model ber-MC = rata-rata lintasan.
- CNN-LSTM tanpa MC (dropout nonaktif) tetap dihitung, tetapi tidak masuk tabel utama; hasilnya
  disimpan terpisah di `evaluasi_<bagian>_cnn_lstm_tanpa_mc.csv` untuk satu kalimat di teks.
- **Tabel B**: PICP dan MPIW ±1σ untuk LSTM-MC Dropout dan CNN-LSTM-MC Dropout × horizon.
- **Tabel C**: metrik per kondisi aliran (< Q1, Q1–Q3, > Q3) untuk semua model; PICP 95% dan MPIW 95%
  hanya untuk model ber-MC.

In [ ]:
NAMA_TABEL = {"Persistence": "Persistence (reference)", "ARIMA": "ARIMA", "CNN": "CNN", "LSTM": "LSTM",
              "LSTM-MC": "LSTM-MC Dropout", "CNN-LSTM-MC": "CNN-LSTM-MC Dropout"}      # baris Tabel A, berurutan
NAMA_TAMPIL = NAMA_TABEL | {"CNN-LSTM": "CNN-LSTM (no MC)"}

def ada(mod, h, k):
    return (k, "mean") in RATA.columns and pd.notna(RATA.loc[(mod, h), (k, "mean")])

def teks(mod, h, k, d):
    m_, s_ = RATA.loc[(mod, h), (k, "mean")], RATA.loc[(mod, h), (k, "std")]
    return f"{m_:.{d}f}" if pd.isna(s_) else f"{m_:.{d}f} ± {s_:.{d}f}"

def nilai(mod, h, k):
    return {f"{k}_mean": RATA.loc[(mod, h), (k, "mean")] if ada(mod, h, k) else np.nan,
            f"{k}_sd": RATA.loc[(mod, h), (k, "std")] if ada(mod, h, k) else np.nan}

def tabel(metrik, model, kondisi=None):
    """(tampilan, csv) baris horizon x model [x kondisi aliran] untuk daftar (kolom, label, desimal)."""
    tampil, csv = [], []
    for h in HORIZON:
        for mod in model:
            for kd in (kondisi or [None]):
                sfx = "" if kd is None else f"_{kd}"
                kunci = {"Horizon": h, "Model": NAMA_TAMPIL[mod]} | (
                    {} if kd is None else {"Flow condition": kd.replace("-", "–").replace("<", "< ").replace(">", "> ")})
                n_ = {"n": int(RATA.loc[(mod, h), (f"n{sfx}", "mean")])} if kd else {}
                tampil.append(kunci | n_ | {lab: teks(mod, h, k + sfx, d) if ada(mod, h, k + sfx) else "—" for k, lab, d in metrik})
                csv.append(kunci | n_ | {f"{k}_{st}": v for k, _, _ in metrik for st, v in
                                         zip(("mean", "sd"), nilai(mod, h, k + sfx).values())})
    idx = ["Horizon", "Model"] + (["Flow condition"] if kondisi else [])
    return pd.DataFrame(tampil).set_index(idx), pd.DataFrame(csv)

# Tabel A: baris model, kolom (horizon, metrik), seperti Tabel 2 Sun dkk. (2026)
M_A = [("NSE", "NSE", 3), ("RMSE", "RMSE (m³/s)", 1), ("MAE", "MAE (m³/s)", 1),
       ("PICP95", "PICP95", 3), ("MPIW95", "MPIW95 (m³/s)", 1)]
TAB_A = pd.DataFrame({(f"{h}-day ahead", lab): [teks(m, h, k, d) if ada(m, h, k) else "—" for m in NAMA_TABEL]
                      for h in HORIZON for k, lab, d in M_A},
                     index=pd.Index(list(NAMA_TABEL.values()), name="Model"))
TAB_A.columns = pd.MultiIndex.from_tuples(TAB_A.columns, names=["Lead time", ""])
tanpa_mc = [NAMA_TABEL[m] for m in NAMA_TABEL if m not in MODEL_MC]
assert TAB_A.shape == (6, 15) and "CNN-LSTM" not in NAMA_TABEL
assert (TAB_A.loc[tanpa_mc, (slice(None), ["PICP95", "MPIW95 (m³/s)"])] == "—").all().all()
assert not (TAB_A.loc[[NAMA_TABEL[m] for m in MODEL_MC]] == "—").any().any()
csv_a = pd.DataFrame([{"model": NAMA_TABEL[m], "horizon": h, **{c: v for k, _, _ in M_A for c, v in nilai(m, h, k).items()}}
                      for m in NAMA_TABEL for h in HORIZON])

# CNN-LSTM tanpa MC: hanya CSV terpisah, dengan CNN-LSTM-MC Dropout sebagai pembanding
csv_det = pd.DataFrame([{"model": "CNN-LSTM (no MC)", "horizon": h,
                         **{c: v for k in ["NSE", "RMSE", "MAE"] for c, v in nilai("CNN-LSTM", h, k).items()},
                         "NSE_CNN_LSTM_MC_mean": RATA.loc[("CNN-LSTM-MC", h), ("NSE", "mean")],
                         "selisih_NSE_MC_kurang_tanpa_MC": RATA.loc[("CNN-LSTM-MC", h), ("NSE", "mean")]
                                                            - RATA.loc[("CNN-LSTM", h), ("NSE", "mean")]}
                        for h in HORIZON])

TAB_B, csv_b = tabel([("PICP68", "PICP ±1σ", 3), ("MPIW68", "MPIW ±1σ (m³/s)", 1)], MODEL_MC)
M_TITIK = [("NSE", "NSE", 3), ("RMSE", "RMSE (m³/s)", 1), ("MAE", "MAE (m³/s)", 1)]
TAB_C, csv_c = tabel(M_TITIK + [("PICP95", "PICP 95%", 3), ("MPIW95", "MPIW 95% (m³/s)", 1)], URUTAN, ["<Q1", "Q1-Q3", ">Q3"])
for nama_, c in [("tabel_A", csv_a), ("tabel_B", csv_b), ("tabel_C", csv_c), ("cnn_lstm_tanpa_mc", csv_det)]:
    c.round(6).to_csv(KELUARAN / f"evaluasi_{BAGIAN}_{nama_}.csv", index=False)
judul = "validation" if BAGIAN == "validasi" else "test"
print(f"Table A — performance on the {judul} set (mean ± sd over 8 seeds; MC Dropout: mean of 1,000 passes, "
      f"form B intervals; — : no prediction interval)")
display(TAB_A)
print(f"CNN-LSTM without MC (separate CSV only): NSE " + ", ".join(
    f"h{h} {csv_det.loc[i, 'NSE_mean']:.3f} (MC Dropout {csv_det.loc[i, 'NSE_CNN_LSTM_MC_mean']:.3f})"
    for i, h in enumerate(HORIZON)))
print(f"\nTable B — ±1σ intervals of MC Dropout models (form B), {judul} set; nominal 0.683")
display(TAB_B)
print(f"Table C — metrics by flow condition (Q1 = {Q1:.1f} m³/s, Q3 = {Q3:.1f} m³/s, training quartiles), {judul} set")
display(TAB_C)

## 9. Gambar

Satu gaya untuk semua gambar: label bahasa Inggris, titik desimal, 300 dpi, dan satu warna tetap per
model (sama dengan tahap 23). Untuk gambar yang menampilkan satu run (b, c, d), setiap model jaringan
diwakili seed yang NSE-nya paling dekat dengan rata-rata 8 seed pada horizon itu; seed wakil ditulis di
gambar.

- **a**: NSE, RMSE, MAE, dan MPIW 95% terhadap horizon, satu garis per model (rata-rata ± sb seed).
- **b**: hidrograf seluruh periode, h1: observasi dan CNN-LSTM-MC dengan rentang 95% dan ±1σ.
- **c**: debit puncak terbesar di periode itu (±15 hari), h1: ARIMA, LSTM-MC, dan CNN-LSTM-MC dengan rentang 95%.
- **d**: observasi terhadap prediksi h1 per model, dengan garis 1:1.
- **e**: PICP 95% per kondisi aliran untuk dua model ber-MC, dengan garis nominal 0,95.

In [ ]:
from cycler import cycler
TEKS, GRID, BATAS = "#333333", "#e3e3e3", "#8c8c8c"
PALET = {"biru": "#3d78a8", "oranye": "#d9822b", "hijau": "#3e9c80", "merah": "#c4664a", "ungu": "#a87a9f", "langit": "#86b8dc"}
ABU_TUA = "#5f5f5f"
WARNA_MODEL = {"Persistence": ABU_TUA, "ARIMA": PALET["ungu"], "CNN": PALET["oranye"], "LSTM": PALET["langit"],
               "LSTM-MC": PALET["hijau"], "CNN-LSTM": PALET["biru"], "CNN-LSTM-MC": PALET["merah"]}
GARIS_MODEL = {m: ("--" if m == "Persistence" else "-") for m in URUTAN}
PENANDA_MODEL = {"Persistence": "o", "ARIMA": "s", "CNN": "^", "LSTM": "v", "LSTM-MC": "D", "CNN-LSTM": "P", "CNN-LSTM-MC": "X"}
WARNA_OBS = "#222222"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "text.color": TEKS, "axes.labelcolor": TEKS, "axes.titlecolor": TEKS, "xtick.color": TEKS, "ytick.color": TEKS,
    "axes.edgecolor": BATAS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "axes.prop_cycle": cycler(color=list(PALET.values())),
    "axes.formatter.use_locale": False,
})
JUDUL_BAGIAN = "validation" if BAGIAN == "validasi" else "test"
GAMBAR = []

def simpan(fig, nama):
    p = FIGUR / f"eval_{BAGIAN}_{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    GAMBAR.append(p)
    plt.show()

def seed_wakil(mod, h):
    """Seed dengan NSE paling dekat dengan rata-rata 8 seed (model jaringan); 0 untuk ARIMA dan persistensi."""
    if mod in ("ARIMA", "Persistence"):
        return 0
    g = PER_RUN[(PER_RUN["model"] == mod) & (PER_RUN["horizon"] == h)].set_index("seed")["NSE"]
    return int((g - g.mean()).abs().idxmin())

def label_wakil(mod, h):
    return mod if mod in ("ARIMA", "Persistence") else f"{mod} (seed {seed_wakil(mod, h)})"
print("seed wakil h1:", {m: seed_wakil(m, 1) for m in URUTAN if m not in ("ARIMA", "Persistence")})

In [ ]:
# Fig. a — metrik terhadap horizon
fig, axs = plt.subplots(1, 4, figsize=(13.5, 3.4))
panel = [("NSE", "NSE (–)", URUTAN), ("RMSE", "RMSE (m³/s)", URUTAN), ("MAE", "MAE (m³/s)", URUTAN),
         ("MPIW95", "MPIW 95% (m³/s)", MODEL_MC)]
for i_, (ax, (k, lab, mods)) in enumerate(zip(axs, panel)):
    for j, mod in enumerate(mods):
        m_ = RATA.loc[mod][(k, "mean")]
        s_ = RATA.loc[mod][(k, "std")].fillna(0)
        ax.errorbar(np.array(HORIZON) + (j - (len(mods) - 1) / 2) * 0.05, m_, yerr=s_, color=WARNA_MODEL[mod],
                    ls=GARIS_MODEL[mod], marker=PENANDA_MODEL[mod], ms=4, lw=1.3, capsize=2, label=mod)
    ax.set_xticks(HORIZON); ax.set_xticklabels([f"{h} d" for h in HORIZON]); ax.set_xlabel("Lead time")
    ax.set_ylabel(lab); ax.set_title(f"({'abcd'[i_]}) {k if k != 'MPIW95' else 'MPIW 95%'}", loc="left")
h_, l_ = axs[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=7, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f"Performance versus lead time, {JUDUL_BAGIAN} set (mean ± sd over 8 seeds)", x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, 0.04, 1, 1))
simpan(fig, "a_metrics_vs_horizon")

In [ ]:
# Fig. b — hidrograf seluruh periode, h1, CNN-LSTM-MC
mod, h = "CNN-LSTM-MC", 1
s_w = seed_wakil(mod, h)
r, E = HASIL_RUN[(mod, h, s_w)], DATA[h]
tgl = pd.DatetimeIndex(E["tanggal"])
def dengan_celah(x, y):
    """Sisipkan NaN di hari tanpa sampel supaya garis dan pita tidak menyambung melewati celah."""
    s = pd.Series(y, index=x)
    return s.reindex(pd.date_range(x[0], x[-1], freq="D"))
fig, ax = plt.subplots(figsize=(13, 4))
lo, hi = dengan_celah(tgl, r["lo95"]), dengan_celah(tgl, r["hi95"])
ax.fill_between(lo.index, lo, hi, color=WARNA_MODEL[mod], alpha=0.18, lw=0, label="95% interval")
s_lo, s_hi = dengan_celah(tgl, r["mu"] - r["sdB"]), dengan_celah(tgl, r["mu"] + r["sdB"])
ax.fill_between(s_lo.index, s_lo, s_hi, color=WARNA_MODEL[mod], alpha=0.35, lw=0, label="±1σ interval")
o_ = dengan_celah(tgl, E["obs"])
ax.plot(o_.index, o_, color=WARNA_OBS, lw=0.8, label="Observed")
p_ = dengan_celah(tgl, r["mu"])
ax.plot(p_.index, p_, color=WARNA_MODEL[mod], lw=0.8, label=f"{mod} mean (seed {s_w})")
m_ = PER_RUN[(PER_RUN["model"] == mod) & (PER_RUN["horizon"] == h) & (PER_RUN["seed"] == s_w)].iloc[0]
ax.text(0.01, 0.97, f"NSE = {m_['NSE']:.3f}   PICP 95% = {m_['PICP95']:.3f}   MPIW 95% = {m_['MPIW95']:.1f} m³/s",
        transform=ax.transAxes, va="top", fontsize=8.5)
ax.set_ylabel("Discharge (m³/s)"); ax.set_xlabel("Date"); ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7])); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.13))
ax.set_title(f"1-day-ahead hydrograph, {JUDUL_BAGIAN} set ({tgl[0].date()} to {tgl[-1].date()})", loc="left")
simpan(fig, "b_hydrograph_h1")

In [ ]:
# Fig. c — kejadian debit puncak terbesar (±15 hari), h1
h, E = 1, DATA[1]
tgl = pd.DatetimeIndex(E["tanggal"])
i_puncak = int(np.argmax(E["obs"]))
t_puncak = tgl[i_puncak]
jendela = (tgl >= t_puncak - pd.Timedelta(days=15)) & (tgl <= t_puncak + pd.Timedelta(days=15))
fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.6), sharey=True)
for ax, mod, huruf in zip(axs, ["ARIMA", "LSTM-MC", "CNN-LSTM-MC"], "abc"):
    s_w = seed_wakil(mod, h)
    r = HASIL_RUN[(mod, h, s_w)]
    x = tgl[jendela]
    ax.fill_between(x, r["lo95"][jendela], r["hi95"][jendela], color=WARNA_MODEL[mod], alpha=0.22, lw=0,
                    label="95% interval" + (" (state-space)" if mod == "ARIMA" else ""))
    ax.plot(x, E["obs"][jendela], color=WARNA_OBS, lw=1.2, marker="o", ms=2.5, label="Observed")
    ax.plot(x, r["pred"][jendela], color=WARNA_MODEL[mod], lw=1.4, label="Prediction" + ("" if mod == "ARIMA" else " (mean)"))
    dalam = np.mean((E["obs"][jendela] >= r["lo95"][jendela]) & (E["obs"][jendela] <= r["hi95"][jendela]))
    ax.text(0.02, 0.97, f"peak obs {E['obs'][i_puncak]:.0f}, pred {r['pred'][i_puncak]:.0f} m³/s\n"
            f"coverage in window {dalam:.2f}", transform=ax.transAxes, va="top", fontsize=8)
    ax.set_title(f"({huruf}) {label_wakil(mod, h)}", loc="left")
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=7)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.legend(loc="upper right", fontsize=7.5)
axs[0].set_ylabel("Discharge (m³/s)")
fig.suptitle(f"Largest peak of the {JUDUL_BAGIAN} set, {t_puncak.date()} (±15 days), 1-day-ahead", x=0.01, ha="left", fontsize=10)
fig.tight_layout()
simpan(fig, "c_peak_event_h1")
print(f"puncak terbesar: {t_puncak.date()}, {E['obs'][i_puncak]:.1f} m3/s; {int(jendela.sum())} hari dalam jendela")

In [ ]:
# Fig. d — scatter observasi terhadap prediksi h1
h, E = 1, DATA[1]
fig, axs = plt.subplots(2, 4, figsize=(13, 6.6))
batas_ = max(float(E["obs"].max()), max(float(HASIL_RUN[(m, h, seed_wakil(m, h))]["pred"].max()) for m in URUTAN)) * 1.03
for ax, mod in zip(axs.ravel(), URUTAN):
    s_w = seed_wakil(mod, h)
    p_ = HASIL_RUN[(mod, h, s_w)]["pred"]
    ax.scatter(E["obs"], p_, s=5, alpha=0.45, color=WARNA_MODEL[mod], lw=0)
    ax.plot([0, batas_], [0, batas_], color=BATAS, lw=1, ls="--", label="1:1")
    m_ = PER_RUN[(PER_RUN["model"] == mod) & (PER_RUN["horizon"] == h) & (PER_RUN["seed"].fillna(0) == s_w)].iloc[0]
    ax.text(0.04, 0.96, f"NSE = {m_['NSE']:.3f}\nRMSE = {m_['RMSE']:.1f} m³/s", transform=ax.transAxes, va="top", fontsize=8)
    ax.set_xlim(0, batas_); ax.set_ylim(0, batas_); ax.set_aspect("equal")
    ax.set_title(label_wakil(mod, h), loc="left", fontsize=9)
for ax in axs[1]:
    ax.set_xlabel("Observed (m³/s)")
for ax in axs[:, 0]:
    ax.set_ylabel("Predicted (m³/s)")
axs.ravel()[-1].axis("off")
fig.suptitle(f"Observed versus 1-day-ahead prediction, {JUDUL_BAGIAN} set", x=0.01, ha="left", fontsize=10)
fig.tight_layout()
simpan(fig, "d_scatter_h1")

In [ ]:
# Fig. e — PICP 95% per kondisi aliran
fig, axs = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
kond = [("<Q1", f"< Q1\n(< {Q1:.0f} m³/s)"), ("Q1-Q3", "Q1–Q3"), (">Q3", f"> Q3\n(> {Q3:.0f} m³/s)")]
x = np.arange(len(kond))
for ax, h in zip(axs, HORIZON):
    for j, mod in enumerate(MODEL_MC):
        m_ = [RATA.loc[(mod, h), (f"PICP95_{k}", "mean")] for k, _ in kond]
        s_ = [RATA.loc[(mod, h), (f"PICP95_{k}", "std")] for k, _ in kond]
        ax.bar(x + (j - 0.5) * 0.36, m_, width=0.34, yerr=s_, capsize=3, color=WARNA_MODEL[mod], label=mod,
               error_kw=dict(lw=0.8, ecolor=TEKS))
    ax.axhline(0.95, color=TEKS, ls="--", lw=1, label="Nominal 0.95")
    ax.set_xticks(x); ax.set_xticklabels([l for _, l in kond]); ax.set_title(f"({'abc'[h - 1]}) {h}-day ahead", loc="left")
y_min = min(0.5, float(np.nanmin([RATA.loc[(m, h), (f"PICP95_{k}", "mean")] for m in MODEL_MC for h in HORIZON for k, _ in kond])) - 0.05)
for ax in axs:
    ax.set_ylim(y_min, 1.02)
axs[0].set_ylabel("PICP 95% (–)")
h_, l_ = axs[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f"95% interval coverage by flow condition, {JUDUL_BAGIAN} set (mean ± sd over 8 seeds; quartiles of the training period)",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=(0, 0.03, 1, 1))
simpan(fig, "e_picp_by_flow_condition")

## Ringkasan

In [ ]:
print(f"bagian: {BAGIAN}; hash konfigurasi: {HASH_KONFIG}")
print(f"sampel per horizon: {N_TABEL['dievaluasi'].to_dict()}; target A:e: {N_TABEL['target A:e'].to_dict()} "
      f"({'dikeluarkan' if BAGIAN == 'uji' else 'tidak dikeluarkan, hanya dilaporkan'})")
print("berkas tabel:", ", ".join(sorted(p.name for p in KELUARAN.glob(f"evaluasi_{BAGIAN}_*.csv"))))
print("berkas gambar:", ", ".join(p.name for p in GAMBAR))
print(f"lintasan MC: {MC_CACHE} ({len(list(MC_CACHE.glob('*.npz')))} berkas)")
print(f"waktu jalan notebook: {(time.time() - MULAI_JALAN) / 60:.1f} menit")